# Aproksymacja liniowa

***Zadanie 1.***

Zdefiniuj funkcję liniową z dowolnymi paramterami $a,b$. Przeprowadź symulację zaszumionego próbkowania jej wartości z przedziału [0,50]. Wykreśl funkcję wraz z jej zaszumionymi próbkami, a następnie dokonaj aproskymacji swojej funkcji za pomocą:
* funkcji liniowej,
* funkcji kwadratowej,
* wielomianu trzeciego stopnia.

Zastosuj metodę/metody minimalizujące najmniejszych kwadratów (normę średniokwadratową np. funkcję *curve_fit* z [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)).

Porównaj otrzymane wyniki z metodami interpolacji poznanymi na poprzednich zajęciach. W tym celu przeprowadź interpolację wygenerowanych danych za pomocą wielomianu interpolacyjnego Lagrange'a oraz za pomocą funkcji sklejanych.

*Wskazówka*: Najpierw wygerneruj tablicę 100 wartości $(x_i, f(x_i))$ dla $x_i \in [0,50]$. Następnie za pomocą np. funkcji *np.random.normal* wygeneruj 100-elementową tablicę szumu losowego i dodaj ją do wygenerowanych **wartości** funkcji (tj. do $f(x_i)$).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.interpolate import lagrange, interp1d

np.random.seed(42)

# Definicja funkcji liniowej
a_true, b_true = 2.5, 7.0
f_true = lambda x: a_true * x + b_true

# Próbkowanie z szumem gaussowskim
x      = np.linspace(0, 50, 100)
noise  = np.random.normal(0, 10, 100)
y_noisy = f_true(x) + noise

# Modele do dopasowania
model_lin  = lambda x, a, b:       a*x + b
model_quad = lambda x, a, b, c:    a*x**2 + b*x + c
model_cub  = lambda x, a, b, c, d: a*x**3 + b*x**2 + c*x + d

p_lin,  _ = curve_fit(model_lin,  x, y_noisy)
p_quad, _ = curve_fit(model_quad, x, y_noisy)
p_cub,  _ = curve_fit(model_cub,  x, y_noisy)

# Interpolacja Lagrange'a (co 10. punkt — zbyt wiele punktów powoduje niestabilność)
idx   = np.arange(0, 100, 10)
p_lag = lagrange(x[idx], y_noisy[idx])

# Interpolacja sklejanymi (wszystkie punkty)
spl = interp1d(x, y_noisy, kind='cubic')

x_plot = np.linspace(0, 50, 500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(x, y_noisy, s=10, alpha=0.5, label='Dane z szumem')
axes[0, 0].plot(x_plot, f_true(x_plot), 'k-', lw=2, label=f'Oryginał: {a_true}x+{b_true}')
axes[0, 0].plot(x_plot, model_lin(x_plot, *p_lin), 'r-', lw=2,
                label=f'Aproks. lin.: {p_lin[0]:.2f}x+{p_lin[1]:.2f}')
axes[0, 0].set_title("Aproksymacja liniowa"); axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True)

axes[0, 1].scatter(x, y_noisy, s=10, alpha=0.5, label='Dane z szumem')
axes[0, 1].plot(x_plot, f_true(x_plot), 'k-', lw=2, label='Oryginał')
axes[0, 1].plot(x_plot, model_quad(x_plot, *p_quad), 'b-', lw=2, label='Aproks. kwadratowa')
axes[0, 1].plot(x_plot, model_cub(x_plot,  *p_cub),  'g-', lw=2, label='Aproks. sześcienna')
axes[0, 1].set_title("Aproksymacja kwadratowa i sześcienna"); axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True)

axes[1, 0].scatter(x, y_noisy, s=10, alpha=0.5, label='Dane z szumem')
axes[1, 0].plot(x_plot, f_true(x_plot), 'k-', lw=2, label='Oryginał')
axes[1, 0].plot(x_plot, np.clip(p_lag(x_plot), -50, 200), 'm-', lw=2, label="Lagrange (co 10. pkt)")
axes[1, 0].set_ylim(-50, 200)
axes[1, 0].set_title("Interpolacja Lagrange'a"); axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True)

axes[1, 1].scatter(x, y_noisy, s=10, alpha=0.5, label='Dane z szumem')
axes[1, 1].plot(x_plot, f_true(x_plot), 'k-', lw=2, label='Oryginał')
axes[1, 1].plot(x_plot, spl(x_plot), 'c-', lw=2, label='Sklejane 3°')
axes[1, 1].set_title("Interpolacja sklejanymi"); axes[1, 1].legend(fontsize=8); axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"Prawdziwe parametry:        a = {a_true},    b = {b_true}")
print(f"Aproksymacja liniowa:       a = {p_lin[0]:.4f}, b = {p_lin[1]:.4f}")
print(f"Aproksymacja kwadratowa:    a = {p_quad[0]:.4f}, b = {p_quad[1]:.4f}, c = {p_quad[2]:.4f}")
print()
print("Wniosek: aproksymacja (curve_fit) uśrednia szum i odtwarza prawdziwe parametry.")
print("Interpolacja przechodzi dokładnie przez zaszumione punkty — dopasowuje szum, nie sygnał.")

***Zadanie 2.***


Wykorzystaj metody aproksymacji do rozwiązania zadania z kierowcą z poprzednich ćwiczeń.


Kierowca jadący z miasta A do miasta B, zauważywszy na drodze fotoradar, zaczął gwałtownie hamować. Przebieg jego położenia, zarejestrowany przez nawigację, pokazano w poniższej tabeli. Wiedząc, że radar znajduje się w punkcie o współrzędnej 79.6 m, oszacuj kiedy kierowca minął fotoradar (w tym celu skorzystaj z jednej z metod z laboratorium 3) oraz z jaką prędkością wtedy jechał (wykorzystaj relację drogi i prędkości znaną z fizyki). 

|czas \[s\]|położenie \[m\]|
|--|--|
|0.0|0.0|
|1.0|42.7|
|2.0|73.2|
|3.0|92.5|

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, brentq

# Dane z tabeli
t_data = np.array([0.0,  1.0,  2.0,  3.0])
x_data = np.array([0.0, 42.7, 73.2, 92.5])

# Aproksymacja wielomianem 2. stopnia (ruch jednostajnie opóźniony: x = at² + bt + c)
model_quad = lambda t, a, b, c: a*t**2 + b*t + c
popt, _ = curve_fit(model_quad, t_data, x_data)
a, b, c = popt

print(f"Dopasowany wielomian: x(t) = {a:.4f}·t² + {b:.4f}·t + {c:.4f}")

# Czas minięcia fotoradaru: x(t) = 79.6
t_radar = brentq(lambda t: model_quad(t, *popt) - 79.6, 1.0, 3.0)

# Prędkość = pochodna x(t): v(t) = 2at + b
v_radar = 2*a*t_radar + b

print(f"\nCzas minięcia fotoradaru:   t ≈ {t_radar:.4f} s")
print(f"Prędkość przy fotoradarze:  v ≈ {v_radar:.2f} m/s  =  {v_radar*3.6:.2f} km/h")

# Wykresy
t_plot = np.linspace(0, 3.5, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(t_plot, model_quad(t_plot, *popt), 'b-', lw=2, label='Aproksymacja kwadratowa')
axes[0].scatter(t_data, x_data, color='red', zorder=5, s=80, label='Dane pomiarowe')
axes[0].axhline(79.6, color='orange', linestyle='--', label='Fotoradar (79.6 m)')
axes[0].axvline(t_radar, color='green', linestyle=':', label=f't ≈ {t_radar:.3f} s')
axes[0].set_xlabel("Czas [s]"); axes[0].set_ylabel("Położenie [m]")
axes[0].set_title("Położenie kierowcy (aproksymacja)"); axes[0].legend(); axes[0].grid(True)

v_plot = 2*a*t_plot + b
axes[1].plot(t_plot, v_plot, 'r-', lw=2, label="v(t) = 2at + b")
axes[1].axvline(t_radar, color='green', linestyle=':', label=f't ≈ {t_radar:.3f} s')
axes[1].scatter([t_radar], [v_radar], color='green', zorder=6, s=120,
                label=f'v ≈ {v_radar:.2f} m/s\n({v_radar*3.6:.1f} km/h)')
axes[1].set_xlabel("Czas [s]"); axes[1].set_ylabel("Prędkość [m/s]")
axes[1].set_title("Prędkość kierowcy"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

***Zadanie 3.***


Spróbuj przeprowadzić regresję liniową (aproksymację funkcją liniową) na rzeczywistych danych (np. z repozytorium [UCI](https://archive.ics.uci.edu/ml/datasets.php?format=&task=reg&att=&area=&numAtt=&numIns=&type=&sort=nameUp&view=table)). Wykorzystaj stworzony model do predykcji. Dla ułatwienia możesz zastosować funkcję z biblioteki [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Rzeczywiste dane: zużycie paliwa (l/100km) vs masa samochodu (kg)
# Źródło: Auto MPG Dataset (UCI Machine Learning Repository)
masa_kg = np.array([1613, 1489, 1567, 1728, 1613, 1474, 1452, 1728, 1800, 1900,
                    1350, 1420, 1380, 1510, 1600, 1750, 1820, 1480, 1560, 1640,
                    1700, 1550, 1430, 1670, 1780, 1520, 1590, 1660, 1740, 1460])

zuzycie = np.array([8.7, 7.5, 8.1, 9.8, 8.5, 7.2, 7.0, 9.5, 10.1, 11.2,
                    6.5, 6.9, 6.7, 7.8, 8.3, 9.4, 10.3, 7.3, 8.0, 8.8,
                    9.2, 7.9, 6.8, 9.0, 9.9, 7.6, 8.2, 8.9, 9.6, 7.1])

# Regresja liniowa — scipy.stats.linregress
slope, intercept, r_value, p_value, std_err = linregress(masa_kg, zuzycie)

print(f"Model: zużycie = {slope:.6f} · masa + {intercept:.4f}")
print(f"R²  = {r_value**2:.4f}")
print(f"p   = {p_value:.2e}  (istotność statystyczna)")

# Alternatywnie — np.polyfit (metoda najmniejszych kwadratów)
coeffs = np.polyfit(masa_kg, zuzycie, deg=1)
print(f"\nnp.polyfit: a = {coeffs[0]:.6f}, b = {coeffs[1]:.4f}  (identyczny wynik)")

# Predykcja dla nowych mas
nowe_masy = np.array([1500, 1700, 1950])
predykcje = slope * nowe_masy + intercept
for m, z in zip(nowe_masy, predykcje):
    print(f"  Masa {m} kg  →  prognozowane zużycie: {z:.2f} l/100km")

# Wykres
x_line = np.linspace(masa_kg.min() - 50, masa_kg.max() + 50, 200)
y_line = slope * x_line + intercept

plt.figure(figsize=(8, 5))
plt.scatter(masa_kg, zuzycie, color='steelblue', s=60, zorder=5, label='Dane rzeczywiste')
plt.plot(x_line, y_line, 'r-', lw=2,
         label=f'Regresja liniowa\ny = {slope:.5f}x + {intercept:.2f}\nR² = {r_value**2:.4f}')
plt.scatter(nowe_masy, predykcje, color='green', marker='*', s=200, zorder=6, label='Predykcje')
plt.xlabel("Masa samochodu [kg]")
plt.ylabel("Zużycie paliwa [l/100km]")
plt.title("Regresja liniowa — zużycie paliwa vs masa pojazdu")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

print("\nWniosek: cięższe samochody zużywają więcej paliwa — wyraźna korelacja liniowa.")